# PHIST shared-kmer threshold vs species-cluster coverage

Sweep PHIST percent-shared-kmer (`Containment`) thresholds and measure the proportion of species clusters with a species-level host prediction, using the consensus logic from `toolkit/bin/uhvdb_phisthost.py`. Then combine with CRISPR species assignments to estimate the maximum joint coverage.

In [1]:
### Load packages
import matplotlib.pyplot as plt
import polars as pl

plt.rcParams.update({'font.size': 14})

In [2]:
### Paths and parameters
PHIST_TSV = '../figure_1/uhgv_hq_hc_results/uhvdb_2026-03-23/uhvdb_phist.tsv.gz'
CRISPR_TSV = '../figure_1/uhgv_hq_hc_results/uhvdb_2026-03-23/uhvdb_crispr.tsv.gz'
SPECIES_INFO = '../figure_1/uhgv_hq_hc_results/uhvdb_2026-03-23/uhvdb_species_info.tsv.gz'

MIN_AGREEMENT = 0.7
CONTAINMENT_THRESHOLDS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

In [3]:
### Load species clusters
species_info = (
    pl.read_csv(SPECIES_INFO, separator='\t')
        .select(['uhvdb_id', 'cluster_id', 'votu_rep'])
)
n_clusters = species_info['cluster_id'].n_unique()
print(f'Total species clusters: {n_clusters}')

Total species clusters: 53595


### PHIST species consensus across containment thresholds

Mirrors `uhvdb_phisthost.py`: for each virus, keep host hits with `Containment >= threshold`, require species-level agreement ≥ 0.7 across all retained hits.

In [4]:
def assign_phist_species_hosts(phist_hits, min_containment=0.2, min_agreement=0.7):
    """Return species-level PHIST host assignments at a containment threshold."""
    filtered = phist_hits.filter(pl.col('Containment') >= min_containment)
    totals = (
        filtered
            .group_by('uhvdb_id')
            .agg(pl.len().alias('total_connections'))
    )
    return (
        filtered
            .filter(pl.col('species_token').is_not_null())
            .group_by(['uhvdb_id', 'species_token'])
            .agg(pl.len().alias('connections'))
            .group_by('uhvdb_id')
            .agg([
                pl.col('connections').max().alias('max_connections'),
                pl.col('species_token').sort_by('connections', descending=True).first().alias('top_taxonomy'),
            ])
            .join(totals, on='uhvdb_id', how='inner')
            .with_columns([
                (pl.col('max_connections') / pl.col('total_connections')).alias('agreement'),
                pl.lit('species').alias('rank'),
                pl.lit(min_containment).alias('containment_threshold'),
            ])
            .filter(pl.col('agreement') >= min_agreement)
    )


def cluster_coverage(hosts, species_info, n_clusters):
    """Percent of species clusters with >=1 genome having a species-level prediction."""
    n_with_pred = (
        species_info
            .join(hosts.select('uhvdb_id').unique(), on='uhvdb_id', how='inner')
            ['cluster_id'].n_unique()
    )
    return n_with_pred, 100 * n_with_pred / n_clusters

In [ ]:
### Load PHIST hits (taxonomy already joined in uhvdb_phist.tsv.gz)
phist_hits = (
    pl.scan_csv(PHIST_TSV, separator='\t')
        .select(['uhvdb_id', 'Containment', 'taxonomy'])
        .with_columns([
            pl.col('Containment').cast(pl.Float64),
            pl.when(pl.col('taxonomy').str.contains(';s__'))
                .then(
                    pl.lit('s__')
                    + pl.col('taxonomy').str.split(';s__').list.get(-1).str.split(';').list.get(0)
                )
                .otherwise(None)
                .alias('species_token'),
        ])
        .with_columns(
            pl.when(pl.col('species_token') == 's__')
                .then(None)
                .otherwise(pl.col('species_token'))
                .alias('species_token')
        )
        .collect(engine='streaming')
)
print(f'PHIST hits: {phist_hits.height:,}')
print(f'Unique viruses with PHIST hits: {phist_hits["uhvdb_id"].n_unique():,}')
print(f'Containment range: {phist_hits["Containment"].min():.3f} - {phist_hits["Containment"].max():.3f}')

PHIST hits: 66,069,674
Unique viruses with PHIST hits: 120,553
Containment range: 0.200 - 1.000


: 

In [ ]:
### Sweep containment thresholds
phist_rows = []
phist_hosts_by_threshold = {}

for thresh in CONTAINMENT_THRESHOLDS:
    hosts = assign_phist_species_hosts(
        phist_hits,
        min_containment=thresh,
        min_agreement=MIN_AGREEMENT,
    )
    phist_hosts_by_threshold[thresh] = hosts
    n_genomes = hosts['uhvdb_id'].n_unique()
    n_clusters_with_pred, pct = cluster_coverage(hosts, species_info, n_clusters)
    phist_rows.append({
        'containment_threshold': thresh,
        'n_genomes_with_species_pred': n_genomes,
        'n_clusters_with_species_pred': n_clusters_with_pred,
        'n_species_clusters': n_clusters,
        'percent_clusters': pct,
    })
    print(
        f'Containment >= {thresh:.2f}: '
        f'{n_clusters_with_pred}/{n_clusters} clusters ({pct:.2f}%), '
        f'{n_genomes:,} genomes'
    )

phist_summary = pl.DataFrame(phist_rows)
phist_summary

Containment >= 0.20: 24672/53595 clusters (46.03%), 61,844 genomes


In [ ]:
### Plot PHIST species-cluster coverage vs shared-kmer threshold
plot_df = phist_summary.to_pandas()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    plot_df['containment_threshold'],
    plot_df['percent_clusters'],
    marker='o',
    color='#2c7fb8',
    linewidth=2,
)
ax.axvline(x=0.2, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('Minimum shared kmer fraction (Containment)', fontdict={'fontweight': 'bold'})
ax.set_ylabel('% species clusters with\nspecies-level PHIST prediction', fontdict={'fontweight': 'bold'})
ax.set_xlim(0.15, 0.95)
ax.set_ylim(0, max(plot_df['percent_clusters'].max() * 1.15, 5))
plt.tight_layout()
plt.show()

### CRISPR species assignments (0 or 1 mismatch)

Reuse the consensus logic from `uhvdb_crisprhost.py` / the CRISPR mismatch notebook.

In [ ]:
def assign_crispr_species_hosts(crispr_tsv, max_mismatches=1, min_agreement=0.7):
    """Return species-level CRISPR host assignments for a mismatch threshold."""
    return (
        pl.scan_csv(crispr_tsv, separator='\t')
            .select(['uhvdb_id', 'species', 'N mismatches'])
            .filter(pl.col('N mismatches') <= max_mismatches)
            .group_by(['uhvdb_id', 'species'])
            .agg(pl.len().alias('connections'))
            .group_by('uhvdb_id')
            .agg([
                pl.col('connections').sum().alias('total_connections'),
                pl.col('connections').max().alias('max_connections'),
                pl.col('species').sort_by('connections', descending=True).first().alias('top_taxonomy'),
            ])
            .with_columns([
                (pl.col('max_connections') / pl.col('total_connections')).alias('agreement'),
                pl.lit('species').alias('rank'),
            ])
            .filter(pl.col('agreement') >= min_agreement)
            .collect(engine='streaming')
    )


crispr_0mm = assign_crispr_species_hosts(CRISPR_TSV, max_mismatches=0, min_agreement=MIN_AGREEMENT)
crispr_1mm = assign_crispr_species_hosts(CRISPR_TSV, max_mismatches=1, min_agreement=MIN_AGREEMENT)
crispr_either = pl.concat([
    crispr_0mm.select('uhvdb_id'),
    crispr_1mm.select('uhvdb_id'),
]).unique()

crispr_rows = []
for label, hosts in [
    ('CRISPR 0 mismatches', crispr_0mm),
    ('CRISPR 1 mismatch', crispr_1mm),
    ('CRISPR either 0 or 1', crispr_either),
]:
    n_clusters_with_pred, pct = cluster_coverage(hosts, species_info, n_clusters)
    crispr_rows.append({
        'method': label,
        'n_clusters_with_species_pred': n_clusters_with_pred,
        'percent_clusters': pct,
    })
    print(f'{label}: {n_clusters_with_pred}/{n_clusters} ({pct:.2f}%)')

crispr_summary = pl.DataFrame(crispr_rows)
crispr_summary

### Combined PHIST + CRISPR coverage

For each PHIST containment threshold, take the union of genomes with a species-level prediction from PHIST or from CRISPR (either 0- or 1-mismatch consensus). Report the maximum joint species-cluster coverage.

In [ ]:
### Union coverage across methods
crispr_clusters = (
    species_info
        .join(crispr_either, on='uhvdb_id', how='inner')
        .select('cluster_id')
        .unique()
)

combined_rows = []
for thresh in CONTAINMENT_THRESHOLDS:
    phist_hosts = phist_hosts_by_threshold[thresh].select('uhvdb_id')
    combined_hosts = pl.concat([phist_hosts, crispr_either]).unique()
    n_clusters_with_pred, pct = cluster_coverage(combined_hosts, species_info, n_clusters)

    phist_clusters = (
        species_info
            .join(phist_hosts, on='uhvdb_id', how='inner')
            .select('cluster_id')
            .unique()
    )
    n_phist_only_clusters = phist_clusters.join(crispr_clusters, on='cluster_id', how='anti').height

    combined_rows.append({
        'containment_threshold': thresh,
        'n_clusters_phist': phist_clusters.height,
        'n_clusters_crispr_either': crispr_clusters.height,
        'n_clusters_combined': n_clusters_with_pred,
        'n_clusters_phist_only': n_phist_only_clusters,
        'percent_clusters_phist': 100 * phist_clusters.height / n_clusters,
        'percent_clusters_crispr_either': 100 * crispr_clusters.height / n_clusters,
        'percent_clusters_combined': pct,
    })
    print(
        f'Containment >= {thresh:.2f}: combined {n_clusters_with_pred}/{n_clusters} '
        f'({pct:.2f}%); PHIST-only clusters vs CRISPR: {n_phist_only_clusters}'
    )

combined_summary = pl.DataFrame(combined_rows)
best = combined_summary.sort('percent_clusters_combined', descending=True).row(0, named=True)
print(
    f"\nMaximum combined coverage: {best['n_clusters_combined']}/{n_clusters} "
    f"({best['percent_clusters_combined']:.2f}%) "
    f"at PHIST Containment >= {best['containment_threshold']:.2f} "
    f"union CRISPR (either 0 or 1 mismatch)"
)
combined_summary

In [ ]:
### Plot PHIST-only vs combined coverage
comb_df = combined_summary.to_pandas()
crispr_pct = comb_df['percent_clusters_crispr_either'].iloc[0]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    comb_df['containment_threshold'],
    comb_df['percent_clusters_phist'],
    marker='o',
    color='#2c7fb8',
    linewidth=2,
    label='PHIST only',
)
ax.plot(
    comb_df['containment_threshold'],
    comb_df['percent_clusters_combined'],
    marker='o',
    color='#d95f0e',
    linewidth=2,
    label='PHIST ∪ CRISPR (0 or 1 mm)',
)
ax.axhline(y=crispr_pct, color='#31a354', linestyle='--', linewidth=1.5, label='CRISPR only (0 or 1 mm)')
ax.axvline(x=0.2, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('Minimum shared kmer fraction (Containment)', fontdict={'fontweight': 'bold'})
ax.set_ylabel('% species clusters with\nspecies-level prediction', fontdict={'fontweight': 'bold'})
ax.set_xlim(0.15, 0.95)
ax.set_ylim(0, max(comb_df['percent_clusters_combined'].max() * 1.15, crispr_pct * 1.15))
ax.legend(frameon=False, fontsize=11)
plt.tight_layout()
plt.show()